# Institutional Ownership-Breadth Signal — Results

Reads the artifacts produced by the pipeline (`results/tables/*.json`,
`results/figures/*.png`) and renders the out-of-sample results. Rebuild the
artifacts with `make all` (or the per-phase CLIs); this notebook only displays
them, so it never leaks look-ahead into the evaluation.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
TABLES, FIGS = ROOT / 'results' / 'tables', ROOT / 'results' / 'figures'
pd.set_option('display.float_format', lambda v: f'{v:.4f}')

## Out-of-sample performance (purged & embargoed CV)

In [ ]:
pd.read_csv(TABLES / 'summary.csv')

## SHAP importance and out-of-sample feature pruning (12m)

In [ ]:
r = json.load(open(TABLES / 'backtest_results_12m.json'))
imp = pd.Series(r['shap_importance']).sort_values(ascending=False)
print('Incremental IC of breadth over size/value baseline:', round(r['incremental_mean_ic'], 4))
print('IC all 16 features:', round(r['ic_all_features'], 4), ' -> pruned:', round(r['ic_pruned'], 4))
print('\nDropped (failed OOS):', r['dropped_features'])
print('Kept:', r['kept_features'])
imp.to_frame('mean_abs_shap')

In [ ]:
for name in ['ic_timeseries_12m', 'decile_spread_12m', 'cumulative_ls_12m', 'shap_importance_12m']:
    p = FIGS / f'{name}.png'
    if p.exists():
        display(Image(filename=str(p)))

## Coverage & data-loss diagnostics

In [ ]:
print('Mapping:', json.load(open(TABLES / 'mapping_coverage.json')))
print('Phase-4:', json.load(open(TABLES / 'phase4_coverage.json')))
d = pd.read_csv(TABLES / 'delisting_dropout.csv')
d = d[(d.horizon_months == 12) & (d.labelable == True)]
print(f"12m within-window delisting dropout (labelable periods): mean {d.dropout_rate.mean()*100:.2f}%")